# 02 — Models

Model definitions for both sentiment classifiers, loaded directly from `backend/app/ml/models.py` -- no logic re-implemented here, only real imports and inspection.

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from wherever this notebook actually lives to find the real
    project root (the folder containing both backend/app/ and data/), so every
    relative path used below resolves correctly regardless of which folder
    this notebook is opened from."""
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the Baseera project root (a folder containing both "
        "backend/app/ and data/) above this notebook's location."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


## CNN2D — from-scratch PyTorch model

Extracted verbatim from the original notebook's cell 131. `Embedding(30000,100,pad_idx=0)` -> 4 parallel Conv2D branches (filter sizes 2/3/4/5, 32 filters each) -> `BatchNorm2d` -> `ReLU` -> adaptive global max-pool -> concat -> `Dropout(0.5)` -> `Linear(128,32)` -> `ReLU` -> `Dropout(0.5)` -> `Linear(32,1)` (raw logit).

In [ ]:
import inspect
from app.ml.models import CNN2DReviewSentiment

print(inspect.getsource(CNN2DReviewSentiment))


In [ ]:
from app.ml.models import load_cnn2d_model
from app.ml.utils import get_device

device = get_device()
cnn_model = load_cnn2d_model("models/cnn2d_review_sentiment.pt", device=device)
n_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"CNN2D loaded on {device} -- {n_params:,} trainable parameters")
# README's own claim: 3,049,345 trainable parameters -- verified live above, not just asserted.


## BERT — fine-tuned transformer

Fine-tuned from `LiYuan/amazon-review-sentiment-analysis` (`bert-base-multilingual-uncased` architecture), `AutoModelForSequenceClassification`, `num_labels=2`. `load_fine_tuned_bert` below has **no fallback path** -- unlike the original notebook's inference cell, which silently re-initialized an untrained head on load failure, this raises instead.

In [ ]:
import inspect
from app.ml.models import load_fine_tuned_bert

print(inspect.getsource(load_fine_tuned_bert))


In [ ]:
bert_model, bert_tokenizer = load_fine_tuned_bert("models/bert_review_sentiment", device=device)
n_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"BERT loaded on {device} -- {n_params:,} trainable parameters")
print("Labels:", bert_model.config.id2label)
